# Оценка качества русскоязычного перефразирования

**Задача:** сгенерировать перефразирование русского предложения и проверить две характеристики: сохранение смысла и отличие формулировки от исходного текста.

**Стек:** PyTorch, Transformers, pandas, NLTK, ROUGE.

В качестве генератора используется `fyaronskiy/ruT5-large-paraphraser`, а семантическая близость оценивается моделью `s-nlp/ruRoberta-large-paraphrase-v1`. BLEU и ROUGE-L считаются относительно эталонного перефразирования, а не исходной фразы.


## План эксперимента

1. Загрузить пары «исходный текст — эталонное перефразирование».
2. Сформировать воспроизводимую тестовую выборку.
3. Сгенерировать по одному варианту для каждого примера.
4. Сравнить генерацию с эталоном по BLEU и ROUGE-L, а с исходным текстом — по семантической близости.

> Для полного запуска желательно использовать GPU: обе модели достаточно крупные.


In [ ]:
import random

import numpy as np
import pandas as pd
import torch
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from rouge_score import rouge_scorer
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_SIZE = 50
print(f"Устройство: {DEVICE}")


In [ ]:
data = pd.read_csv(
    "hf://datasets/cointegrated/ru-paraphrase-NMT-Leipzig/train.csv",
    usecols=["original", "ru"],
).dropna()

data = data.rename(columns={"original": "source", "ru": "reference"})
data = data.drop_duplicates(subset=["source", "reference"]).reset_index(drop=True)

_, test_data = train_test_split(data, test_size=0.20, random_state=SEED)
evaluation_data = test_data.sample(
    n=min(SAMPLE_SIZE, len(test_data)), random_state=SEED
).reset_index(drop=True)

print(f"Всего пар: {len(data):,}")
print(f"Примеров для оценки: {len(evaluation_data):,}")
evaluation_data.head(3)


In [ ]:
GENERATOR_NAME = "fyaronskiy/ruT5-large-paraphraser"
SIMILARITY_NAME = "s-nlp/ruRoberta-large-paraphrase-v1"

generator_tokenizer = AutoTokenizer.from_pretrained(GENERATOR_NAME)
generator_model = AutoModelForSeq2SeqLM.from_pretrained(GENERATOR_NAME).to(DEVICE)
generator_model.eval()

similarity_tokenizer = AutoTokenizer.from_pretrained(SIMILARITY_NAME)
similarity_model = AutoModelForSequenceClassification.from_pretrained(
    SIMILARITY_NAME
).to(DEVICE)
similarity_model.eval()


In [ ]:
def paraphrase(text: str, num_beams: int = 5) -> str:
    encoded = generator_tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        max_length=256,
    ).to(DEVICE)

    input_length = encoded["input_ids"].shape[1]
    with torch.inference_mode():
        output = generator_model.generate(
            **encoded,
            num_beams=num_beams,
            max_new_tokens=min(256, int(input_length * 1.5) + 16),
            no_repeat_ngram_size=2,
            early_stopping=True,
        )
    return generator_tokenizer.decode(output[0], skip_special_tokens=True)


def semantic_similarity(text_a: str, text_b: str) -> float:
    encoded = similarity_tokenizer(
        str(text_a),
        str(text_b),
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(DEVICE)

    with torch.inference_mode():
        logits = similarity_model(**encoded).logits
    return torch.softmax(logits, dim=-1)[0, 1].item()


rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
smoothing = SmoothingFunction().method4


def reference_metrics(reference: str, prediction: str) -> tuple[float, float]:
    reference_tokens = str(reference).split()
    prediction_tokens = str(prediction).split()
    bleu = sentence_bleu(
        [reference_tokens], prediction_tokens, smoothing_function=smoothing
    )
    rouge_l = rouge.score(str(reference), str(prediction))["rougeL"].fmeasure
    return bleu, rouge_l


In [ ]:
rows = []
for row in evaluation_data.itertuples(index=False):
    prediction = paraphrase(row.source)
    bleu, rouge_l = reference_metrics(row.reference, prediction)
    rows.append(
        {
            "source": row.source,
            "reference": row.reference,
            "prediction": prediction,
            "bleu": bleu,
            "rouge_l": rouge_l,
            "semantic_similarity": semantic_similarity(row.source, prediction),
        }
    )

results = pd.DataFrame(rows)
results[["bleu", "rouge_l", "semantic_similarity"]].agg(["mean", "std"]).round(3)


In [ ]:
pd.set_option("display.max_colwidth", 120)
results.sort_values("semantic_similarity", ascending=False).head(5)


## Интерпретация

- Высокая семантическая близость означает, что смысл исходной фразы в основном сохранён.
- BLEU и ROUGE-L показывают сходство с одним конкретным эталоном и могут занижать качество допустимых альтернативных формулировок.
- Для продуктовой оценки полезно дополнить автоматические метрики ручной проверкой смысла, грамматики и разнообразия.
